In [2]:
!pip install requests beautifulsoup4 nltk transformers sentencepiece torch networkx scikit-learn

In [3]:
import nltk
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
import requests
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize
import re

url = "https://www.usatoday.com/story/news/politics/2025/06/13/pete-hegseth-pentagon-invade-greenland-plan/84188458007/"

headers = {
    "User-Agent": "DATA622-homework-bot/1.0 (gm97457@umbc.edu)"
}

resp = requests.get(url, headers=headers, timeout=20)
html = resp.text

soup = BeautifulSoup(html, "html.parser")
text = soup.get_text(separator=" ")
text = re.sub(r"\s+", " ", text).strip()

print("Original text (first 700 chars):\n")
print(text[:700])

sentences = sent_tokenize(text)
print("\nNumber of sentences in article:", len(sentences))

Original text (first 700 chars):

Hegseth says Pentagon has 'contingency' plan to invade Greenland Find us on Google 📌 Eating like it is 1776 Start the day smarter ☀️ Get the USA TODAY app U.S. Politics Sports Entertainment Life Money Travel Opinion Crossword ONLY AT USA TODAY: Newsletters For Subscribers Real Estate From the Archives USA TODAY PLAY Crossword eNewspaper Magazines Investigations Video Podcasts Humankind WITNESS (True Crime) California Atlanta Chicago Just Curious Best-selling Booklist Top Workplaces Nomination Legals Press Releases Cars Grocery Shopping OUR PORTFOLIO: Real Estate 10BEST USAT Savings Shopping Blueprint WITNESS (True Crime) Southern Kitchen Home Internet Services POLITICS Greenland Add Topic He

Number of sentences in article: 19


In [6]:
import numpy as np
import networkx as nx
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

stop_words = set(stopwords.words("english"))

def clean_sentence(s):
    return s.replace("\n", " ").strip()

clean_sents = [clean_sentence(s) for s in sentences]

# TF-IDF sentence vectors
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(clean_sents)  # (num_sents, vocab)

# Cosine similarity matrix
sim_matrix = (tfidf_matrix * tfidf_matrix.T).toarray()

# Build graph and run PageRank
nx_graph = nx.from_numpy_array(sim_matrix)
scores = nx.pagerank(nx_graph)   # scores is a dict: {node_index: score}

# Rank sentences by their PageRank score
ranked = sorted(
    ((scores[i], i, s) for i, s in enumerate(clean_sents)),
    reverse=True
)

# Choose top N sentences (e.g., 5)
N = min(5, len(clean_sents))
top_indices = sorted([idx for (_, idx, _) in ranked[:N]])
textrank_summary = " ".join(clean_sents[i] for i in top_indices)

print("\n1) TextRank extractive summary:\n")
print(textrank_summary)


1) TextRank extractive summary:

Politics Sports Entertainment Life Money Travel Opinion Crossword ONLY AT USA TODAY: Newsletters For Subscribers Real Estate From the Archives USA TODAY PLAY Crossword eNewspaper Magazines Investigations Video Podcasts Humankind WITNESS (True Crime) California Atlanta Chicago Just Curious Best-selling Booklist Top Workplaces Nomination Legals Press Releases Cars Grocery Shopping OUR PORTFOLIO: Real Estate 10BEST USAT Savings Shopping Blueprint WITNESS (True Crime) Southern Kitchen Home Internet Services POLITICS Greenland Add Topic Hegseth says Pentagon has many 'contingencies' in Greenland - including invading it Cybele Mayes-Osterman USA TODAY June 13, 2025, 2:33 p.m. ET Hear this story Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island. Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to i

In [7]:
import string
from collections import Counter
import nltk

words = nltk.word_tokenize(text.lower())
words = [w for w in words if w.isalpha() and w not in stop_words]

freq = Counter(words)

sent_scores = []
for s in clean_sents:
    tokens = nltk.word_tokenize(s.lower())
    score = sum(freq.get(w, 0) for w in tokens)
    sent_scores.append(score)

N = min(5, len(clean_sents))
top_indices_freq = sorted(np.argsort(sent_scores)[-N:])
freq_summary = " ".join(clean_sents[i] for i in top_indices_freq)

print("\n2) Frequency-based extractive summary:\n")
print(freq_summary)


2) Frequency-based extractive summary:

Hegseth says Pentagon has 'contingency' plan to invade Greenland Find us on Google 📌 Eating like it is 1776 Start the day smarter ☀️ Get the USA TODAY app U.S. Politics Sports Entertainment Life Money Travel Opinion Crossword ONLY AT USA TODAY: Newsletters For Subscribers Real Estate From the Archives USA TODAY PLAY Crossword eNewspaper Magazines Investigations Video Podcasts Humankind WITNESS (True Crime) California Atlanta Chicago Just Curious Best-selling Booklist Top Workplaces Nomination Legals Press Releases Cars Grocery Shopping OUR PORTFOLIO: Real Estate 10BEST USAT Savings Shopping Blueprint WITNESS (True Crime) Southern Kitchen Home Internet Services POLITICS Greenland Add Topic Hegseth says Pentagon has many 'contingencies' in Greenland - including invading it Cybele Mayes-Osterman USA TODAY June 13, 2025, 2:33 p.m. Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are pl

In [15]:
from transformers import pipeline

max_chars = 3000
input_text = text[:max_chars]

bart_gen = pipeline(
    "text-generation",
    model="facebook/bart-large-cnn",
    tokenizer="facebook/bart-large-cnn"
)

prompt_bart = (
    "Summarize the following news article in 3-4 sentences, "
    "focusing on the main political points:\n\n" + input_text
)

bart_output = bart_gen(
    prompt_bart,
    max_new_tokens=200,
    do_sample=False,
    num_return_sequences=1,
)

abs_summary = bart_output[0]["generated_text"].split(
    "focusing on the main political points:"
)[-1].strip()

print("\n3) Abstractive summary (BART via text-generation):\n")
print(abs_summary)

Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

BartForCausalLM LOAD REPORT from: facebook/bart-large-cnn
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.bias     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.weight   | UNEXPECTED |  | 
model.encoder.laye


3) Abstractive summary (BART via text-generation):

Hegseth says Pentagon has 'contingency' plan to invade Greenland Find us on Google 📌 Eating like it is 1776 Start the day smarter ☀️ Get the USA TODAY app U.S. Politics Sports Entertainment Life Money Travel Opinion Crossword ONLY AT USA TODAY: Newsletters For Subscribers Real Estate From the Archives USA TODAY PLAY Crossword eNewspaper Magazines Investigations Video Podcasts Humankind WITNESS (True Crime) California Atlanta Chicago Just Curious Best-selling Booklist Top Workplaces Nomination Legals Press Releases Cars Grocery Shopping OUR PORTFOLIO: Real Estate 10BEST USAT Savings Shopping Blueprint WITNESS (True Crime) Southern Kitchen Home Internet Services POLITICS Greenland Add Topic Hegseth says Pentagon has many 'contingencies' in Greenland - including invading it Cybele Mayes-Osterman USA TODAY June 13, 2025, 2:33 p.m. ET Hear this story Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" i

In [9]:
lead3_summary = " ".join(sentences[:3])
print("\n4) Lead-3 summary (first three sentences):\n")
print(lead3_summary)


4) Lead-3 summary (first three sentences):

Hegseth says Pentagon has 'contingency' plan to invade Greenland Find us on Google 📌 Eating like it is 1776 Start the day smarter ☀️ Get the USA TODAY app U.S. Politics Sports Entertainment Life Money Travel Opinion Crossword ONLY AT USA TODAY: Newsletters For Subscribers Real Estate From the Archives USA TODAY PLAY Crossword eNewspaper Magazines Investigations Video Podcasts Humankind WITNESS (True Crime) California Atlanta Chicago Just Curious Best-selling Booklist Top Workplaces Nomination Legals Press Releases Cars Grocery Shopping OUR PORTFOLIO: Real Estate 10BEST USAT Savings Shopping Blueprint WITNESS (True Crime) Southern Kitchen Home Internet Services POLITICS Greenland Add Topic Hegseth says Pentagon has many 'contingencies' in Greenland - including invading it Cybele Mayes-Osterman USA TODAY June 13, 2025, 2:33 p.m. ET Hear this story Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenl

In [10]:
import math

num_sents = len(sentences)
k = max(1, math.ceil(0.2 * num_sents))  # about 20%
manual_comp_summary = " ".join(sentences[:k])

print(f"\n5) Manual compression summary (~20% of {num_sents} sentences → {k} sentences):\n")
print(manual_comp_summary)


5) Manual compression summary (~20% of 19 sentences → 4 sentences):

Hegseth says Pentagon has 'contingency' plan to invade Greenland Find us on Google 📌 Eating like it is 1776 Start the day smarter ☀️ Get the USA TODAY app U.S. Politics Sports Entertainment Life Money Travel Opinion Crossword ONLY AT USA TODAY: Newsletters For Subscribers Real Estate From the Archives USA TODAY PLAY Crossword eNewspaper Magazines Investigations Video Podcasts Humankind WITNESS (True Crime) California Atlanta Chicago Just Curious Best-selling Booklist Top Workplaces Nomination Legals Press Releases Cars Grocery Shopping OUR PORTFOLIO: Real Estate 10BEST USAT Savings Shopping Blueprint WITNESS (True Crime) Southern Kitchen Home Internet Services POLITICS Greenland Add Topic Hegseth says Pentagon has many 'contingencies' in Greenland - including invading it Cybele Mayes-Osterman USA TODAY June 13, 2025, 2:33 p.m. ET Hear this story Defense Secretary Pete Hegseth said the Pentagon has plans for multiple 

In [16]:
prompt_llm = (
    "Write a concise, student-friendly summary (2-3 sentences) "
    "of this article:\n\n" + input_text
)

llm_output = bart_gen(
    prompt_llm,
    max_new_tokens=120,
    do_sample=False,
    num_return_sequences=1,
)

llm_summary = llm_output[0]["generated_text"].split(
    "this article:"
)[-1].strip()

print("\n6) LLM (BART) summary of the article:\n")
print(llm_summary)

Both `max_new_tokens` (=120) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



6) LLM (BART) summary of the article:

Hegseth says Pentagon has 'contingency' plan to invade Greenland Find us on Google 📌 Eating like it is 1776 Start the day smarter ☀️ Get the USA TODAY app U.S. Politics Sports Entertainment Life Money Travel Opinion Crossword ONLY AT USA TODAY: Newsletters For Subscribers Real Estate From the Archives USA TODAY PLAY Crossword eNewspaper Magazines Investigations Video Podcasts Humankind WITNESS (True Crime) California Atlanta Chicago Just Curious Best-selling Booklist Top Workplaces Nomination Legals Press Releases Cars Grocery Shopping OUR PORTFOLIO: Real Estate 10BEST USAT Savings Shopping Blueprint WITNESS (True Crime) Southern Kitchen Home Internet Services POLITICS Greenland Add Topic Hegseth says Pentagon has many 'contingencies' in Greenland - including invading it Cybele Mayes-Osterman USA TODAY June 13, 2025, 2:33 p.m. ET Hear this story Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland –